# Data import to csv

In [1]:
import sys
sys.path.append(r'D:\Python\main-scripts')

import pandas as pd
from connectors import OracleConnector

# ============================================
# CONFIGURATION
# ============================================
TABLE_NAME = 'JAPAN_CURVE_RATES'
MARKET = 'DE'  # German yield curve

# ============================================
# READ DATA
# ============================================

# Connect to database
connection = OracleConnector(dsn='mopdb', user=None, pwd=None)
print("✓ Database connection established\n")

# SQL query - pull German zero yields only (excluding PAR yields)
query = f"""
SELECT 
    REFERENCE_DATE,
    PRICE_SOURCE,
    CURVE,
    ZERO_1Y,
    ZERO_2Y,
    ZERO_3Y,
    ZERO_4Y,
    ZERO_5Y,
    ZERO_6Y,
    ZERO_7Y,
    ZERO_8Y,
    ZERO_9Y,
    ZERO_10Y,
    ZERO_12Y,
    ZERO_15Y,
    ZERO_20Y,
    ZERO_25Y,
    ZERO_30Y
FROM BMI_LAB.{TABLE_NAME}
WHERE MARKET = '{MARKET}' AND CURVE = 'LOWESS' AND PRICE_SOURCE ='IBOXX'
ORDER BY REFERENCE_DATE
"""

# Load data into pandas DataFrame
df = connection.read_sql(query)

# Print summary
print(f"✓ Loaded {len(df)} rows from {TABLE_NAME}")
print(f"✓ Market: {MARKET} (German yield curve)")
print(f"✓ Date range: {df['REFERENCE_DATE'].min()} to {df['REFERENCE_DATE'].max()}\n")

# Display first few rows
print("First 5 rows:")
print(df.head())

print("\n" + "="*80)
print("DataFrame Info:")
print("="*80)
print(df.info())

print("\n" + "="*80)
print("Sample of data:")
print("="*80)
print(df)

# Close connection
connection._cnxn.close()
print("\n✓ Database connection closed")

✓ Database connection established

✓ Loaded 7397 rows from JAPAN_CURVE_RATES
✓ Market: DE (German yield curve)
✓ Date range: 1999-01-02 00:00:00 to 2026-04-20 00:00:00

First 5 rows:
  REFERENCE_DATE PRICE_SOURCE   CURVE   ZERO_1Y   ZERO_2Y   ZERO_3Y   ZERO_4Y  \
0     1999-01-02        IBOXX  LOWESS  0.030762  0.031195  0.031997  0.032941   
1     1999-01-03        IBOXX  LOWESS  0.030762  0.031195  0.031997  0.032941   
2     1999-01-04        IBOXX  LOWESS  0.029998  0.030520  0.031295  0.032191   
3     1999-01-05        IBOXX  LOWESS  0.029571  0.030313  0.031213  0.032194   
4     1999-01-06        IBOXX  LOWESS  0.029700  0.030397  0.031287  0.032263   

    ZERO_5Y   ZERO_6Y   ZERO_7Y   ZERO_8Y   ZERO_9Y  ZERO_10Y  ZERO_12Y  \
0  0.034222  0.035681  0.037116  0.038044  0.038508  0.038488  0.038439   
1  0.034222  0.035681  0.037116  0.038044  0.038508  0.038488  0.038439   
2  0.033384  0.034750  0.036075  0.036989  0.037488  0.037549  0.037667   
3  0.033414  0.034794  0.03613

In [2]:
# ============================================
# SAVE YIELD CURVE DATA TO CSV
# ============================================

import os

# Define save path
save_path = r"Data\Source"

# Create filename using MARKET variable
filename = f"{MARKET}_YieldCurve.csv"

# Full filepath
filepath = os.path.join(save_path, filename)

# Drop source names
df_yieldcurve = df.drop(columns=['PRICE_SOURCE','CURVE'])

# Drop weekends
weekends = df_yieldcurve[df_yieldcurve['REFERENCE_DATE'].dt.dayofweek >= 5]
if len(weekends) > 0:
    print("="*60)
    print(f"DROPPED WEEKENDS: {len(weekends)} dates")
    print("="*60)
    for date in weekends['REFERENCE_DATE']:
        print(f"  {date.strftime('%Y-%m-%d')} ({date.strftime('%A')})")
    print()

df_yieldcurve = df_yieldcurve[df_yieldcurve['REFERENCE_DATE'].dt.dayofweek < 5].reset_index(drop=True)

# Drop rows with any missing zero coupon yields
zero_columns = ['ZERO_1Y', 'ZERO_2Y', 'ZERO_3Y', 'ZERO_4Y', 'ZERO_5Y',
                'ZERO_6Y', 'ZERO_7Y', 'ZERO_8Y', 'ZERO_9Y', 'ZERO_10Y',
                'ZERO_12Y', 'ZERO_15Y', 'ZERO_20Y', 'ZERO_25Y', 'ZERO_30Y']

nan_rows = df_yieldcurve[df_yieldcurve[zero_columns].isnull().any(axis=1)]
if len(nan_rows) > 0:
    print("="*60)
    print(f"DROPPED MISSING YIELDS: {len(nan_rows)} dates")
    print("="*60)
    for _, row in nan_rows.iterrows():
        missing_cols = [c for c in zero_columns if pd.isna(row[c])]
        print(f"  {row['REFERENCE_DATE'].strftime('%Y-%m-%d')} ({row['REFERENCE_DATE'].strftime('%A')}) — missing: {', '.join(missing_cols)}")
    print()

df_yieldcurve = df_yieldcurve.dropna(subset=zero_columns).reset_index(drop=True)

# Save to CSV
df_yieldcurve.to_csv(filepath, index=False)

print(f"✓ Saved: {filename}")
print(f"  Location: {save_path}")
print(f"  Rows: {len(df_yieldcurve)}")

DROPPED WEEKENDS: 405 dates
  1999-01-02 (Saturday)
  1999-01-03 (Sunday)
  1999-01-09 (Saturday)
  1999-01-10 (Sunday)
  1999-01-16 (Saturday)
  1999-01-17 (Sunday)
  1999-01-23 (Saturday)
  1999-01-24 (Sunday)
  1999-01-30 (Saturday)
  1999-01-31 (Sunday)
  1999-02-06 (Saturday)
  1999-02-07 (Sunday)
  1999-02-13 (Saturday)
  1999-02-14 (Sunday)
  1999-02-20 (Saturday)
  1999-02-21 (Sunday)
  1999-02-27 (Saturday)
  1999-02-28 (Sunday)
  1999-03-06 (Saturday)
  1999-03-07 (Sunday)
  1999-03-13 (Saturday)
  1999-03-14 (Sunday)
  1999-03-20 (Saturday)
  1999-03-21 (Sunday)
  1999-03-27 (Saturday)
  1999-03-28 (Sunday)
  1999-04-03 (Saturday)
  1999-04-04 (Sunday)
  1999-04-10 (Saturday)
  1999-04-11 (Sunday)
  1999-04-17 (Saturday)
  1999-04-18 (Sunday)
  1999-04-24 (Saturday)
  1999-04-25 (Sunday)
  1999-05-01 (Saturday)
  1999-05-02 (Sunday)
  1999-05-08 (Saturday)
  1999-05-09 (Sunday)
  1999-05-15 (Saturday)
  1999-05-16 (Sunday)
  1999-05-22 (Saturday)
  1999-05-23 (Sunday)
  1999

# IV data

In [3]:
# ============================================
# IV DATA SETUP
# ============================================
import pandas as pd
import os

# ============================================
# CONFIGURATION
# ============================================
MARKET = 'DE'
save_path = r"Data\Source"

# ============================================
# IMPORT DATA
# ============================================

# Import yield curve dates
yc_path = os.path.join(save_path, f"{MARKET}_YieldCurve.csv")
df_yc = pd.read_csv(yc_path)
df_yc['REFERENCE_DATE'] = pd.to_datetime(df_yc['REFERENCE_DATE'], format='ISO8601')

# Import IV data (no header)
iv_path = r"C:\Users\westenb\Downloads\IVSDW.csv"
df_iv = pd.read_csv(iv_path, header=None, names=['REFERENCE_DATE', 'IV'])
df_iv['REFERENCE_DATE'] = pd.to_datetime(df_iv['REFERENCE_DATE'], format='%d/%m/%Y')

# ============================================
# FILTER TO YIELD CURVE DATES
# ============================================

yc_dates = df_yc['REFERENCE_DATE']
df_iv_filtered = yc_dates.to_frame().merge(
    df_iv, on='REFERENCE_DATE', how='left'
)

# Report missing dates before interpolation
missing = df_iv_filtered[df_iv_filtered['IV'].isna()]['REFERENCE_DATE']
if len(missing) > 0:
    print("="*60)
    print(f"LINEAR INTERPOLATION APPLIED: {len(missing)} dates")
    print("="*60)
    for date in missing:
        print(f"  {date.strftime('%Y-%m-%d')} ({date.strftime('%A')})")
    print()
else:
    print("✓ No missing IV dates — no interpolation needed\n")

# Interpolate missing values
df_iv_filtered['IV'] = df_iv_filtered['IV'].interpolate(method='linear', limit_direction='both')

# ============================================
# SAVE
# ============================================

# Match date format of YieldCurve.csv
df_iv_filtered['REFERENCE_DATE'] = df_iv_filtered['REFERENCE_DATE'].dt.strftime('%Y-%m-%d')

iv_out = os.path.join(save_path, "IV.csv")
df_iv_filtered.to_csv(iv_out, index=False)

print(f"✓ Saved: IV.csv")
print(f"  Location: {save_path}")
print(f"  Rows: {len(df_iv_filtered)}")
print(f"  Date range: {df_iv_filtered['REFERENCE_DATE'].min()} to {df_iv_filtered['REFERENCE_DATE'].max()}")

LINEAR INTERPOLATION APPLIED: 95 dates
  2000-12-25 (Monday)
  2001-12-25 (Tuesday)
  2002-02-11 (Monday)
  2002-03-29 (Friday)
  2005-04-14 (Thursday)
  2006-11-02 (Thursday)
  2006-11-13 (Monday)
  2006-12-01 (Friday)
  2006-12-12 (Tuesday)
  2006-12-18 (Monday)
  2006-12-19 (Tuesday)
  2006-12-22 (Friday)
  2007-01-08 (Monday)
  2007-05-07 (Monday)
  2008-04-24 (Thursday)
  2008-05-05 (Monday)
  2008-05-26 (Monday)
  2008-08-25 (Monday)
  2008-09-17 (Wednesday)
  2009-01-09 (Friday)
  2009-05-19 (Tuesday)
  2009-05-20 (Wednesday)
  2009-05-25 (Monday)
  2009-07-02 (Thursday)
  2009-07-09 (Thursday)
  2009-08-10 (Monday)
  2009-08-21 (Friday)
  2009-09-22 (Tuesday)
  2009-09-29 (Tuesday)
  2009-10-01 (Thursday)
  2009-10-02 (Friday)
  2009-12-28 (Monday)
  2010-01-26 (Tuesday)
  2010-02-17 (Wednesday)
  2010-02-18 (Thursday)
  2010-03-03 (Wednesday)
  2010-05-21 (Friday)
  2010-05-31 (Monday)
  2010-06-22 (Tuesday)
  2010-09-13 (Monday)
  2010-09-16 (Thursday)
  2010-09-17 (Friday)
 